# Intermittent Sales — Scoring Notebook

This notebook loads the trained model from Google Drive and generates a ranked customer
contact list for the sales team.

**Prerequisites:**
- Notebook 1 (model_training.ipynb) must have been executed at least once
- The trained model must be saved in Google Drive at `Intermitent_sales_model_ML/`
- Input data must follow the required feature schema (see README)

In [1]:
import joblib
import json
import numpy as np
import pandas as pd
from tensorflow import keras
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Load path — must match SAVE_PATH from Notebook 1
LOAD_PATH = '/content/drive/MyDrive/Intermitent_sales_model_ML/'

# Load model, scaler and metadata
model   = keras.models.load_model(LOAD_PATH + 'best_model.keras')
scaler  = joblib.load(LOAD_PATH + 'scaler.pkl')

with open(LOAD_PATH + 'model_metadata.json', 'r') as f:
    metadata = json.load(f)

THRESHOLD    = metadata['threshold']
feature_cols = metadata['feature_cols']
model_name   = metadata['model_name']

print(f"Model loaded    : {model_name}")
print(f"Threshold       : {THRESHOLD}")
print(f"Features expected: {len(feature_cols)}")
print(f"\nFeature list:")
for col in feature_cols:
    print(f"  - {col}")

Mounted at /content/drive
Model loaded    : Deep Learning
Threshold       : 0.43
Features expected: 10

Feature list:
  - Discount
  - Sales_Orders_Lag_1
  - Sales_Orders_Lag_2
  - Sales_Orders_Lag_3
  - months_since_last_sale
  - Activity_Ratio
  - Activity_Rate_3M
  - Activity_Recency_Interaction
  - Season_Activity_Interaction
  - Num_SalesOrders_L3M


## Input Data

Load your dataset with the engineered features ready.
The DataFrame must contain exactly the columns listed above and no more, no less.
Column names and order must match the training schema.

In [2]:
import pandas as pd
from google.colab import files
import io


# Upload file manually in Colab
uploaded = files.upload()
df_new = pd.read_csv(io.BytesIO(list(uploaded.values())[0]))

# Validate that all required features are present
missing = [col for col in feature_cols if col not in df_new.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

print(f"Data loaded     : {df_new.shape[0]} customers, {df_new.shape[1]} columns")
print("All required features present.")

Saving sample_input.csv to sample_input (1).csv
Data loaded     : 10 customers, 11 columns
All required features present.


## Scoring

The model generates a purchase probability for each customer.
Customers are ranked from highest to lowest probability and assigned a priority tier.

| Priority | Probability | Recommended Action |
|---|---|---|
| 🔴 Very High | > 0.80 | Contact immediately |
| 🟠 High | 0.60 – 0.80 | Schedule a touchpoint this month |
| 🟡 Medium | threshold – 0.60 | Include in email campaign |
| ⚪ Low | < threshold | No action needed this month |

In [3]:
# Align columns to training feature order
X_new = df_new[feature_cols].values

# Scale using the same scaler fitted during training
X_scaled = scaler.transform(X_new)

# Generate purchase probabilities
probabilities = model.predict(X_scaled).ravel()

# Build output table
output = df_new[['keyId']].copy()
output['purchase_probability'] = probabilities.round(3)
output['contact_flag']         = (probabilities >= THRESHOLD).astype(int)

# Assign priority tiers
output['priority'] = pd.cut(
    probabilities,
    bins=[0, THRESHOLD, 0.60, 0.80, 1.01],
    labels=['Low', 'Medium', 'High', 'Very High']
)

# Recommended action per tier
action_map = {
    'Very High' : 'Contact immediately',
    'High'      : 'Schedule a touchpoint this month',
    'Medium'    : 'Include in email campaign',
    'Low'       : 'No action needed this month',
}
output['recommended_action'] = output['priority'].map(action_map)

# Rank by probability
output = output.sort_values('purchase_probability', ascending=False).reset_index(drop=True)
output.index += 1  # rank starts at 1

print(f"Total customers scored : {len(output)}")
print(f"Flagged for contact    : {output['contact_flag'].sum()}")
print(f"Threshold applied      : {THRESHOLD}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 169ms/step
Total customers scored : 10
Flagged for contact    : 9
Threshold applied      : 0.43


## Output

In [4]:
# Preview ranked contact list
display(output.head(20))

# Export to CSV for the sales team
output.to_csv('customer_contact_list.csv', index_label='Rank')
print("Contact list saved to customer_contact_list.csv")

,keyId,purchase_probability,contact_flag,priority,recommended_action
1,HUNGO,0.696,1,High,Schedule a touchpoint this month
2,ERNSH,0.692,1,High,Schedule a touchpoint this month
3,WHITC,0.636,1,High,Schedule a touchpoint this month
4,BONAP,0.618,1,High,Schedule a touchpoint this month
5,DRACD,0.557,1,Medium,Include in email campaign
6,LETSS,0.547,1,Medium,Include in email campaign
7,ALFKI,0.476,1,Medium,Include in email campaign
8,QUICK,0.462,1,Medium,Include in email campaign
9,FOLKO,0.452,1,Medium,Include in email campaign
10,SAVEA,0.423,0,Low,No action needed this month


Contact list saved to customer_contact_list.csv
